# U01_C1 — Sequences as Text
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 1:** Bioinformatics and Internet
**Type:** Course notebook · **Time:** 35 min · **Difficulty:** ★☆☆
**Goes with:** Lectures L1–L2, Practical S1 (Linux and Sequence Basics)
**Before this:** None

## 1. What you will learn
- **Treat** biological sequences as standard Python strings with 0-based indexing and slicing [LO1].
- **Calculate** fundamental nucleotide statistics including length, base counts, and GC content [LO1].
- **Read** FASTA formatted files directly from disk using Biopython's `SeqIO` module [LO1].
- **Generate** reverse complements and understand biological strand orientation [LO1].


## 2. Biological question
Human hemoglobin subunit beta (*HBB*, RefSeq accession `NM_000518.5`) encodes the 147-amino acid beta-globin chain. Mutations in this gene cause sickle-cell anemia and beta-thalassemia. When bioinformaticians receive a new sequence from a sequencing facility or database, the very first step is quality checking: How long is the transcript? What is its base composition? Where does the coding sequence begin, and what is its reverse complement? In this notebook, we explore how Python represents biological sequences as text strings.


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U01_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Sequences as Strings — Indexing and Slicing
### Theory
In Python, a sequence of DNA or protein is naturally represented as a string of characters (`str`). Python uses **0-based indexing**: the first nucleotide is at index `0`, the second at `1`, and so on. Negative indices count backward from the end (`-1` is the last character). Slicing syntax `seq[start:stop]` extracts characters from `start` up to but *excluding* `stop`. Slicing with a step `seq[start:stop:step]` allows stepping through bases, and `seq[::-1]` reverses a string.


In [ ]:
# Worked Example: Slicing the 5' leader of HBB mRNA
# The first 50 nucleotides of HBB mRNA contain the 5' untranslated region (5' UTR)
hbb_5utr = "ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC"

print("Length of 5' UTR:", len(hbb_5utr), "nucleotides")
print("First nucleotide (index 0):", hbb_5utr[0])
print("Last nucleotide (index -1):", hbb_5utr[-1])
print("First 10 nucleotides (slice 0:10):", hbb_5utr[0:10])
print("Next 10 nucleotides (slice 10:20):", hbb_5utr[10:20])


In [ ]:
# Try it: Change the slice indices below to inspect nucleotides 20 to 35
sample_slice = hbb_5utr[20:35]
print("Slice [20:35]:", sample_slice)
print("Length of this slice:", len(sample_slice))


**Common mistakes:**
- *Off-by-one errors:* In Python, `seq[0:3]` returns 3 characters (indices 0, 1, 2), NOT indices 0, 1, 2, 3.
- *1-based confusion:* Biology papers and GenBank flat files use 1-based coordinates. Base 1 in GenBank is index 0 in Python!


## 4. Concept 2: Counting Nucleotides and GC Content
### Theory
The GC content (fraction of guanine and cytosine bases) is a primary biophysical feature of genomes and genes. GC pairs form three hydrogen bonds, whereas AT pairs form two. GC-rich regions have higher melting temperatures ($T_m$) and distinct codon usage patterns. In mammals, exons typically have higher GC content than introns. GC percentage is defined as:

$$\text{GC}\% = \frac{\text{Count}(G) + \text{Count}(C)}{\text{Total Length}} \times 100$$


In [ ]:
# Worked Example: Calculating base counts and GC content
seq = "ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACCATGGTGCACCTG"

count_a = seq.count("A")
count_c = seq.count("C")
count_g = seq.count("G")
count_t = seq.count("T")
total = len(seq)

gc_percent = ((count_g + count_c) / total) * 100

print(f"Base counts: A={count_a}, C={count_c}, G={count_g}, T={count_t} (Total={total})")
print(f"GC content: {gc_percent:.2f}%")


In [ ]:
# Try it: Calculate AT percentage using the counts above
at_percent = ((count_a + count_t) / total) * 100
print(f"AT content: {at_percent:.2f}%")
print(f"Check sum: {gc_percent + at_percent:.1f}%")


**Common mistakes:**
- *Case sensitivity:* `.count("g")` will return 0 if the string is uppercase `"G"`. Always ensure case consistency with `.upper()`.
- *Ambiguous bases:* Some sequencing reads contain `N` or IUPAC degeneracy codes. Dividing by `count_a + count_c + count_g + count_t` avoids dividing by ambiguous bases.


## 5. Concept 3: Reading FASTA Files with Biopython SeqIO
### Theory
FASTA is the standard text-based format for nucleotide and peptide sequences. A FASTA record starts with a single-line header beginning with `>`, followed by sequence lines. Biopython's `Bio.SeqIO` module provides unified parsers:
- `SeqIO.read(filename, "fasta")`: reads a single record from a file (raises an exception if the file contains more or fewer than one record).
- `SeqIO.parse(filename, "fasta")`: returns an iterator yielding `SeqRecord` objects for multi-FASTA files.
Each `SeqRecord` object contains `.id`, `.description`, and `.seq`.


In [ ]:
# Worked Example: Loading human HBB mRNA from data/
from Bio import SeqIO

fasta_file = DATA_DIR / "sequences" / "hbb_human_mrna.fasta"
record = SeqIO.read(fasta_file, "fasta")

print("Record ID:", record.id)
print("Record Description:", record.description)
print("Sequence Length:", len(record.seq), "bp")
print("First 60 bp of mRNA:\n", record.seq[:60])


In [ ]:
# Try it: Load the coding sequence (CDS) of HBB and compare lengths
cds_file = DATA_DIR / "sequences" / "hbb_human_cds.fasta"
cds_record = SeqIO.read(cds_file, "fasta")

print("CDS ID:", cds_record.id)
print("CDS Length:", len(cds_record.seq), "bp")
print("Difference (UTR length):", len(record.seq) - len(cds_record.seq), "bp")


**Common mistakes:**
- Calling `SeqIO.read()` on a multi-sequence FASTA file. `SeqIO.read()` requires exactly one sequence. For multiple sequences, use `for record in SeqIO.parse(handle, "fasta"):`.
- Treating `record.seq` as an ordinary string for operations where Biopython methods are needed (or forgetting `str(record.seq)` when passing to standard regex).


## 6. Concept 4: Complement and Reverse Complement
### Theory
DNA is double-stranded and antiparallel. One strand runs 5' $\to$ 3' and the complementary strand runs 3' $\to$ 5'. When writing DNA sequences, molecular biologists always write sequences in the 5' to 3' direction. Therefore, finding the complementary strand sequence requires both complementing the bases ($A \leftrightarrow T, G \leftrightarrow C$) and reversing the order. Biopython provides `.reverse_complement()` directly on `Seq` objects.


In [ ]:
# Worked Example: Reverse complement of the first 21 bp of HBB CDS
from Bio.Seq import Seq

coding_start = cds_record.seq[:21]
rev_comp = coding_start.reverse_complement()

print("Original 5' -> 3':", coding_start)
print("Complement:        ", coding_start.complement())
print("Rev Complement:    ", rev_comp)


## 7. Compare with the Web Tool
In Practical S1, we inspected sequence records on NCBI and used command-line tools like EMBOSS `geecee`.
Let's compute the overall GC percentage of the complete 626 bp HBB mRNA and compare it with the NCBI GenBank metadata:


In [ ]:
# Calculate full HBB mRNA GC%
full_seq = record.seq
gc_hbb = ((full_seq.count("G") + full_seq.count("C")) / len(full_seq)) * 100
at_hbb = ((full_seq.count("A") + full_seq.count("T")) / len(full_seq)) * 100

print(f"HBB mRNA (NM_000518.5) Length: {len(full_seq)} bp")
print(f"Python calculated GC%: {gc_hbb:.2f}%")
print(f"Python calculated AT%: {at_hbb:.2f}%")
print("EMBOSS geecee reported GC%: 51.76%")


## 8. Summary
- Biological sequences in Python are strings of characters, indexed from 0 to $N-1$.
- `len(seq)` gives the sequence length; `seq.count('G')` counts individual base occurrences.
- `Bio.SeqIO.read()` parses a single FASTA record into a `SeqRecord` object containing `.id` and `.seq`.
- `.reverse_complement()` generates the reverse-complement strand in standard 5' $\to$ 3' orientation.

| Function / Method | Module / Type | What it does | Example |
|---|---|---|---|
| `len()` | Built-in | Returns number of elements | `len(record.seq)` |
| `str.count()` | `str` / `Seq` | Counts occurrences of substring | `seq.count("G")` |
| `SeqIO.read()` | `Bio.SeqIO` | Reads exactly one record | `SeqIO.read(path, "fasta")` |
| `SeqIO.parse()` | `Bio.SeqIO` | Iterates over multiple records | `SeqIO.parse(path, "fasta")` |
| `.reverse_complement()` | `Bio.Seq.Seq` | Inverts orientation & complements | `seq.reverse_complement()` |


## 9. Next Steps
- Open `U01_P_Sequences_as_Text.ipynb` to complete hands-on practice exercises.
- Self-test your understanding with `U01_Q_Bioinformatics_and_Internet.ipynb`.
